# Silver Layer — Data Cleaning & Feature Engineering

**Project:** Bank Customer Churn Analytics (Medallion Architecture)  
**Notebook:** `02_silver_transformation`  
**Author:** Harshkumar Patel  
**Last Updated:** June 29, 2026

---

## Overview

This notebook implements the **Silver layer** of the Medallion Architecture.

The Bronze dataset is cleaned, standardized, and enriched to produce analytics-ready data. Customer records are then organized into a Star Schema optimized for downstream reporting and Power BI.

---

## Responsibilities

- Load the Bronze Delta table
- Remove unnecessary and sensitive columns
- Standardize column names and data types
- Eliminate duplicate records
- Engineer business-focused features
- Build the Star Schema
- Persist the transformed tables in Delta Lake

> **Silver Principle:** Clean, validate, and enrich the data while preserving row-level granularity. No business aggregations are performed.

---

## Source

| Property | Value |
|----------|-------|
| Database | `churn_bronze` |
| Table | `raw_bank_customers` |
| Records | 80,000 |

---

## Target Tables

| Table | Purpose |
|--------|---------|
| `fact_churn_events` | Core customer churn facts |
| `dim_customer` | Customer demographics |
| `dim_scores` | Engagement and risk metrics |
| `dim_customer_features` | Engineered business features |

---

## Pipeline Flow

```
Bronze Table
      │
      ▼
Clean & Standardize
      │
      ▼
Feature Engineering
      │
      ▼
Build Star Schema
      │
      ▼
Silver Delta Tables
```

## Step 1 — Load Bronze Data

Load the Bronze Delta table into a Spark DataFrame to begin the transformation process.

In [0]:
# loading data from the bronze delta table into silver for cleaning
df_bronze = spark.table("churn_bronze.raw_bank_customers")
print(f"Rows loaded from bronze table: {df_bronze.count():,}")

## Step 2 — Remove Unnecessary Columns

Remove personally identifiable information (PII) and pipeline metadata that are not required for analytical workloads.

### Columns Removed

- Personal identifiers
- Pipeline metadata
- Analysis-irrelevant fields

In [0]:
# Check the number of columns before cleaning
print(f"Number of Columns: {len(df_bronze.columns)}")
# drop columns that are not needed for the analysis
df_clean = df_bronze.drop('full_name', 'address','_ingestion_timestamp', '_source_file', '_pipeline_layer')
print(f"Number of Columns after dropping: {len(df_clean.columns)}")

%md
### Dropping Irrelevant Columns

dropping columns that are not needed for churn analysis to reduce complexity
and keep the star schema clean

columns dropped and why:
- married — not relevant to churn behaviour
- last_active_date, last_transaction_month, created_date — date columns not needed for this analysis
- risk_segment — replaced by our own churn_risk_label
- digital_behavior, cluster_group — not useful for our KPIs
- loyalty_level — replaced by our own engagement_tier

In [0]:
df_clean = df_clean.drop(
    'married',
    'last_active_date',
    'last_transaction_month',
    'created_date',
    'risk_segment',
    'digital_behavior',
    'cluster_group',
    'loyalty_level'
)

print(f"columns remaining: {len(df_clean.columns)}")
print(df_clean.columns)

## Step 3 — Standardize the Dataset

Apply consistent naming conventions and correct data types to improve readability and maintainability.

In [0]:
df_clean = df_clean.withColumnRenamed('tenure_ye', 'tenure_year')
df_clean = df_clean.withColumnRenamed('credit_sco', 'credit_score')
df_clean = df_clean.withColumnRenamed('monthly_ir', 'monthly_income')
df_clean = df_clean.withColumnRenamed('nums_card', 'num_of_cards')
df_clean = df_clean.withColumnRenamed('nums_service', 'num_of_services')

print(df_clean.columns)

## Step 4 — Data Quality Improvements

Perform basic quality checks before feature engineering.

### Validation

- Remove duplicate customer records
- Verify key data types

In [0]:
# Count records before removing duplicates
print(df_clean.count())
# Remove duplicate customer IDs
df_clean = df_clean.dropDuplicates(['id'])
# Verify record count after cleaning
print(df_clean.count())

In [0]:
from pyspark.sql.types import DoubleType
# Converting it to double because the column contains decimal values ot the balance can be in decimal values.
df_clean = df_clean.withColumn('balance', df_clean['balance'].cast(DoubleType()))

print(df_clean.printSchema())

## Step 5 — Standardize Categorical Values

Normalize categorical values to improve consistency across downstream analytics.

### Standardizations

- Translate occupation values to English
- Standardize business terminology
- Validate distinct category values

In [0]:
# translating occupation values from Vietnamese to English
df_clean = df_clean.withColumn('occupation',
    when(col('occupation') == 'Nội trợ/Sinh viên', 'Housewife/Student')
    .when(col('occupation') == 'Lao động phổ thông', 'General Labour')
    .when(col('occupation') == 'Hưu trí', 'Retired')
    .when(col('occupation') == 'Nhân viên văn phòng/Công chức', 'Office Worker/Civil Servant')
    .when(col('occupation') == 'Giáo viên/Giảng viên', 'Teacher/Lecturer')
    .when(col('occupation') == 'Kinh doanh/Bán hàng', 'Business/Sales')
    .when(col('occupation') == 'Kế toán/Tài chính', 'Accounting/Finance')
    .when(col('occupation') == 'Kỹ sư/Chuyên viên IT', 'Engineer/IT Specialist')
    .when(col('occupation') == 'Quản lý/Lãnh đạo', 'Manager/Executive')
    .when(col('occupation') == 'Chủ Doanh nghiệp nhỏ', 'Small Business Owner')
    .otherwise('Other')
)

# verify
df_clean.select('occupation').distinct().display()

## Step 6 — Feature Engineering

Create business-oriented features that improve reporting and segmentation.

### Features Created

| Feature | Description |
|----------|-------------|
| `age_group` | Customer age category |
| `balance_tier` | Balance segmentation |
| `tenure_group` | Customer tenure category |
| `engagement_tier` | Customer engagement level |
| `churn_risk_label` | Behavioral churn risk |
| `churn_reason_inferred` | Inferred churn driver |

### Age Group & Balance Tier

Group customers into age categories for easier analysis.

Categorize customers based on account balance.

In [0]:
from pyspark.sql.functions import when

#Analyising the age column
print(df_clean.select('age').summary().display())
#Dividing age into 3 catogories
df_clean = df_clean.withColumn('age_group',
    when(df_clean['age'] <= 30, 'Young')
    .when(df_clean['age'] <= 50, 'Middle Aged')
    .otherwise('Senior')
)

df_clean.select('age', 'age_group').display()

#Analyising the balance column
print(df_clean.select('balance').summary().display())

#Dividing balance into 4 catogories

#important(
# dividing balance into tiers based on quartiles (25th, 50th, 75th percentile)
# this ensures customers are evenly distributed across tiers making analysis more meaningful)


df_clean = df_clean.withColumn('balance_tier',
    when(df_clean['balance'] <= 14622087, 'Low')
    .when(df_clean['balance'] <= 30896619, 'Medium')
    .when(df_clean['balance'] <= 69591695, 'High')
    .otherwise('Premium')
)


       
df_clean.select('balance', 'balance_tier').display()



### Tenure Group

Group customers based on how long they have been with the bank.

In [0]:
#Analyising the tenure column
print(df_clean.select('tenure_year').summary().display())
#Dividing tenure into 3 catogories
df_clean = df_clean.withColumn('tenure_group',
            when(df_clean['tenure_year'] <= 1, 'New')
            .when(df_clean['tenure_year'] <= 3, 'Regular')
            .otherwise('Loyal'))

df_clean.select('tenure_year', 'tenure_group').display()

df_clean.select('age', 'age_group').display()
df_clean.select('balance', 'balance_tier').display()
df_clean.select('tenure_year', 'tenure_group').display()

### Engagement & Churn Risk

Create engagement levels and a simple churn risk label using business rules.

In [0]:
#Analyising the Engagement column
df_clean.select('engagement_score').summary().display()

#Dividing engagement into 3 catogories
df_clean = df_clean.withColumn('engagement_tier',
            when(df_clean['engagement_score'] <= 25, 'Low')
            .when(df_clean['engagement_score'] <= 50, 'Medium')
            .otherwise('High'))

df_clean.select('engagement_score', 'engagement_tier').display()

In [0]:
#Analyising the risk_segment column
df_clean.select('risk_segment').distinct().display()
#not using this a sit oly has value low and medium

df_clean = df_clean.withColumn('churn_risk_label',
    when((df_clean['active_member'] == False) & (df_clean['engagement_tier'] == 'Low'), 'High Risk')
    .when((df_clean['active_member'] == True) & (df_clean['engagement_tier'] == 'High'), 'Low Risk')
    .otherwise('Medium Risk')
)

df_clean.select('churn_risk_label').display()


## Design Decision — Behavioral Churn Risk

The source dataset includes `risk_score` and `risk_segment`, which represent general financial risk.

For customer retention analysis, a separate `churn_risk_label` is derived using customer activity and engagement behavior. This behavioral feature provides a more relevant indicator for churn analysis and complements the existing financial risk metrics.

In [0]:
print(df_clean.columns)
print(f"total columns: {len(df_clean.columns)}")

In [0]:
df_clean.select("risk_score","risk_segment","churn_risk_label").display()


#Important 
###Addition of churn_reason_inferred

In [0]:
# inferring churn reason from behavioural signals
# since dataset doesnt have explicit churn reasons
# we derive them from patterns in the data

from pyspark.sql.functions import col


df_clean = df_clean.withColumn('churn_reason_inferred',
    # CHURNED customers - assign actual reason
    when((col('exit') == True) & (col('active_member') == False) & (col('engagement_tier') == 'Low'), 'Churned - Disengagement')
    .when((col('exit') == True) & (col('tenure_group') == 'New'), 'Churned - Poor Onboarding')
    .when((col('exit') == True) & (col('balance_tier') == 'Premium'), 'Churned - Better Offer Elsewhere')
    .when((col('exit') == True) & (col('credit_score') < 500), 'Churned - Financial Stress')
    .when((col('exit') == True) & (col('num_of_services') <= 1), 'Churned - Low Product Engagement')
    .when((col('exit') == True), 'Churned - Other')
    
    # HIGH RISK retained customers - assign risk reason
    .when((col('exit') == False) & (col('churn_risk_label') == 'High Risk') & (col('active_member') == False), 'At Risk - Disengagement')
    .when((col('exit') == False) & (col('churn_risk_label') == 'High Risk') & (col('engagement_tier') == 'Low'), 'At Risk - Low Engagement')
    .when((col('exit') == False) & (col('churn_risk_label') == 'High Risk'), 'At Risk - Behavioural Signals')
    
    # LOW/MEDIUM RISK retained customers
    .otherwise('Retained')
)

df_clean.select('churn_reason_inferred').distinct().display()

## Step 7 — Build the Star Schema

The transformed dataset is decomposed into a Star Schema to improve query performance and simplify downstream analytics.

### Tables Created

| Table | Description |
|--------|-------------|
| `fact_churn_events` | Customer churn facts |
| `dim_customer` | Demographic attributes |
| `dim_scores` | Engagement and risk metrics |
| `dim_customer_features` | Engineered categorical features |

In [0]:
# selecting columns for fact table - the core churn event
df_fact = df_clean.select(
    'id',
    'balance',
    'monthly_income',
    'tenure_year',
    'num_of_cards',
    'num_of_services',
    'active_member',
    'exit'
)

df_fact.write.format("delta").mode("overwrite").saveAsTable("churn_silver.fact_churn_events")

print(f"fact_churn_events rows: {df_fact.count():,}")

In [0]:
df_dim_customer = df_clean.select(
    "id",
    "credit_score",
    "gender",
    "age",
    "occupation",
    "origin_province",
    "customer_segment"
)

df_dim_customer.write.format("delta").mode("overwrite").saveAsTable("churn_silver.dim_customer")

print(f"dim_customer rows: {df_dim_customer.count():,}")

In [0]:
df_dim_scores = df_clean.select(
    "id",
    "engagement_score",
    "risk_score"
)

df_dim_scores.write.format("delta").mode("overwrite").saveAsTable("churn_silver.dim_scores")

print(f"dim_scores rows: {df_dim_scores.count():,}")

In [0]:
# rewriting dim_customer_features with churn_reason_inferred included
df_features = df_clean.select(
    'id',
    'age_group',
    'balance_tier',
    'tenure_group',
    'engagement_tier',
    'churn_risk_label',
    'churn_reason_inferred'
)

df_features.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("churn_silver.dim_customer_features")

print(f"updated rows: {df_features.count():,}")
print(df_features.columns)

## Silver Layer Summary

| Metric | Result |
|--------|--------|
| Source Records | 80,000 |
| Duplicate Records Removed | 0 |
| PII Columns Removed | ✓ |
| Features Engineered | 6 |
| Star Schema Tables | 4 |
| Storage Format | Delta Lake |

### Output Tables

- `churn_silver.fact_churn_events`
- `churn_silver.dim_customer`
- `churn_silver.dim_scores`
- `churn_silver.dim_customer_features`

### Status

Silver transformation completed successfully.

**Next Notebook:** `03_gold_business_layer`